# Vizualizace exaktního řešení: efekty hyperboloidálních transformací krok za krokem

Žádná numerická evoluce — vyhodnocujeme **exaktní** odcházející gaussovský balík a díváme se,
co s ním postupně dělají jednotlivé transformace hyperboloidálního přístupu. Dva případy:

**1D (rovinná vlna)**, $\partial_T^2\psi = \partial_x^2\psi$: čistě odcházející řešení
$$\psi_{1\mathrm{D}}(T, x) = f(T - x), \qquad f(u) = e^{-(u + R_0)^2/\sigma^2},$$
amplituda se při šíření nemění.

**3D (sférická vlna)**, $\partial_T^2\psi = \partial_R^2\psi + \frac{2}{R}\partial_R\psi$:
$$\psi_{3\mathrm{D}}(T, R) = \frac{f(T-R) - f(T+R)}{R},$$
regulární v počátku ($\chi = R\psi$ lichá), s $1/R$ úpadkem amplitudy; pro balík z $R_0 > 0$
je v $R > 0$ prakticky čistě odcházející, $\chi \approx f(T-R)$.

**Mezikroky transformace** (aplikujeme postupně):

1. **kompaktifikace** $R = r/\Omega$, $\Omega = 1 - r^2/S^2$ — $\mathcal{I}^+$ se dostane na konečné $r = S$,
2. **hyperboloidální čas** $t = T - h(R)$, $h(R) = \sqrt{S^2 + R^2} - S$ — balík dorazí na $\mathcal{I}^+$ v konečném čase,
3. **přeškálování** $\chi = R\,\psi$ (jen 3D) — zruší $1/R$ úpadek; v 1D není co škálovat, $\chi \equiv \psi$.

Pro vyhodnocení v $(t, r)$ potřebujeme retardovaný/advancovaný čas v uzavřené formě
(regulární i na scri, kde $R \to \infty$); s $Q = \sqrt{S^2\Omega^2 + r^2}$:

$$
u = T - R = t - k(r), \qquad k(r) = R - h(R) = S - \frac{S^2\,\Omega}{r + Q},
$$
$$
v = T + R = t + \frac{r + Q}{\Omega} - S \;\;(\to \infty \text{ na scri, } f(v) \to 0).
$$

In [ ]:
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
# --- Parametry balíku a kompaktifikace ---
R0 = 4.0    # střed balíku (v retardovaném čase u = -R0)
sigma = 0.5 # šířka balíku
S = 10.0    # poloha scri v kompaktifikované souřadnici


def f(u):
    return np.exp(-np.power((u + R0) / sigma, 2))


def fprime(u):
    return -2 * (u + R0) / sigma**2 * f(u)


# --- Kompaktifikace a výšková funkce (vše regulární včetně r = S) ---
def compact_helpers(r):
    Omega = 1 - (r / S) ** 2
    Q = np.sqrt(S**2 * Omega**2 + r**2)
    R_of_r = np.divide(r, Omega, out=np.full_like(r, np.inf), where=Omega > 0)
    k_r = S - S**2 * Omega / (r + Q)  # k = R - h(R), k(0) = 0, k(S) = S
    v_geom = np.divide(r + Q, Omega, out=np.full_like(r, np.inf), where=Omega > 0) - S  # (T+R) - t
    return Omega, Q, R_of_r, k_r, v_geom


# ================= 1D (rovinná vlna) =================
def psi1_phys(T, x):
    return f(T - x)


def psi1_comp(T, R_of_r):
    return f(T - R_of_r)  # na scri f(-inf) = 0


def psi1_hyp(t, k_r):
    return f(t - k_r)     # regulární všude, na scri f(t - S)

# v 1D je chi = psi (žádné přeškálování není potřeba)


# ================= 3D (sférická vlna) =================
def chi3_phys(T, R):
    return f(T - R) - f(T + R)


def psi3_phys(T, R):
    """psi = chi / R, v počátku l'Hospital: psi(T, 0) = -2 f'(T)."""
    R_safe = np.where(R > 1e-12, R, 1.0)
    return np.where(R > 1e-12, chi3_phys(T, R) / R_safe, -2 * fprime(T))


def chi3_comp(T, R_of_r):
    return f(T - R_of_r) - f(T + R_of_r)  # na scri 0 - 0 = 0


def psi3_comp(T, R_of_r):
    chi = chi3_comp(T, R_of_r)
    R_safe = np.where((R_of_r > 1e-12) & np.isfinite(R_of_r), R_of_r, 1.0)
    psi = np.where(np.isfinite(R_of_r), chi / R_safe, 0.0)
    return np.where(R_of_r > 1e-12, psi, -2 * fprime(T))


def chi3_hyp(t, k_r, v_geom):
    return f(t - k_r) - f(t + v_geom)


def psi3_hyp(t, R_of_r, k_r, v_geom):
    chi = chi3_hyp(t, k_r, v_geom)
    R_safe = np.where((R_of_r > 1e-12) & np.isfinite(R_of_r), R_of_r, 1.0)
    psi = np.where(np.isfinite(R_of_r), chi / R_safe, 0.0)  # na scri psi -> 0
    return np.where(R_of_r > 1e-12, psi, -2 * fprime(t))    # počátek: h(0) = 0, tedy T = t

## Prostoročasové diagramy: mezikroky transformace

Každý sloupec = jeden krok navíc; nahoře 1D, dole 3D. Přerušovaně v prvním sloupci
hyperboloidální řezy $t = \mathrm{konst}$.

- **1. sloupec** — fyzikální $(x/R,\, T)$: balík letí po nulové přímce.
- **2. sloupec** — jen kompaktifikace $(r, T)$: scri je na konečném $r = S$, ale stopa balíku
  se zvedá do svislé asymptoty (na scri v konečném $T$ nikdy nedorazí) a nekonečně se zužuje —
  proto se kompaktifikace bez změny časové funkce nedělá.
- **3. sloupec** — + hyperboloidální čas $(r, t)$: stopa je zase „diagonální" a protne
  $\mathcal{I}^+$ v konečném čase $t = S - R_0$. V 3D ale $\psi$ na scri vyhasíná ($1/R$).
- **4. sloupec** — + přeškálování $\chi = R\psi$: signál je řádu jedna až na scri.
  V 1D není co škálovat ($\chi \equiv \psi$), čtvrtý panel je totožný s třetím.

In [ ]:
# --- Heatmapy: 2 řádky (1D, 3D) x 4 sloupce (mezikroky) ---
nT, nx = 600, 600
T_grid = np.linspace(0, 10, nT)
x_grid = np.linspace(0, 10, nx)
r_grid = np.linspace(0, S, nx)

Omega_g, Q_g, R_of_r_g, k_g, v_geom_g = compact_helpers(r_grid)
Tc = T_grid[:, None]

panels = [
    # (řádek, sloupec, data, osa x, popisek x, titulek, vmax)
    (0, 0, psi1_phys(Tc, x_grid[None, :]), x_grid, "$x$", "1D: $\\psi(T, x)$ — fyzikální", 1.0),
    (0, 1, psi1_comp(Tc, R_of_r_g[None, :]), r_grid, "$r$", "1D: $\\psi(T, r)$ — jen kompaktifikace", 1.0),
    (0, 2, psi1_hyp(Tc, k_g[None, :]), r_grid, "$r$", "1D: $\\psi(t, r)$ — + hyp. čas", 1.0),
    (0, 3, psi1_hyp(Tc, k_g[None, :]), r_grid, "$r$", "1D: $\\chi \\equiv \\psi$ — škálování netřeba", 1.0),
    (1, 0, psi3_phys(Tc, x_grid[None, :]), x_grid, "$R$", "3D: $\\psi(T, R)$ — fyzikální", 0.25),
    (1, 1, psi3_comp(Tc, R_of_r_g[None, :]), r_grid, "$r$", "3D: $\\psi(T, r)$ — jen kompaktifikace", 0.25),
    (1, 2, psi3_hyp(Tc, R_of_r_g[None, :], k_g[None, :], v_geom_g[None, :]), r_grid, "$r$", "3D: $\\psi(t, r)$ — + hyp. čas", 0.25),
    (1, 3, chi3_hyp(Tc, k_g[None, :], v_geom_g[None, :]), r_grid, "$r$", "3D: $\\chi(t, r) = R\\psi$ — + přeškálování", 1.0),
]

fig, axes = plt.subplots(2, 4, figsize=(19, 8.5), sharey=True)

for row, col, data, x, xlab, title, vmax in panels:
    ax = axes[row, col]
    im = ax.pcolormesh(x, T_grid, data, cmap="viridis", vmin=0, vmax=vmax, rasterized=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlab)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    if col > 0:
        ax.axvline(S, color="w", lw=1.5, linestyle=":")
        ax.text(S - 0.15, 9.2, "$\\mathcal{I}^+$", color="w", ha="right", fontsize=11)

# hyperboloidální řezy t = konst ve fyzikálních diagramech
R_line = np.linspace(0, 10, 300)
h_line = np.sqrt(S**2 + R_line**2) - S
for row in (0, 1):
    for t0 in [0, 2, 4, 6, 8]:
        axes[row, 0].plot(R_line, t0 + h_line, "w--", lw=1, alpha=0.8)
    axes[row, 0].set_ylim(0, 10)

axes[0, 0].set_ylabel("$T$, resp. $t$")
axes[1, 0].set_ylabel("$T$, resp. $t$")
plt.tight_layout()
plt.show()

## Animace 1D: mezikroky

Horní řada běží ve fyzikálním čase $T$, dolní v hyperboloidálním $t$ (stejná hodnota parametru).
V 1D si balík drží amplitudu všude — po přidání hyperboloidálního času dorazí na $\mathcal{I}^+$
s plnou jedničkou i bez škálování.

In [ ]:
# --- ANIMACE 1D ---
x_anim = np.linspace(0, 10, 1001)
r_anim = np.linspace(0, S, 1001)
s_frames = np.linspace(0, 10, 81)

Omega_a, Q_a, R_of_r_a, k_a, v_geom_a = compact_helpers(r_anim)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
titles_1d = [
    "$\\psi(T, x)$ — fyzikální",
    "$\\psi(T, r)$ — jen kompaktifikace",
    "$\\psi(t, r)$ — + hyp. čas",
    "$\\chi \\equiv \\psi$ — škálování netřeba",
]
lines_1d = []
for ax, title in zip(axes.flat, titles_1d):
    (ln,) = ax.plot(x_anim, np.zeros_like(x_anim), lw=2, color="firebrick")
    lines_1d.append(ln)
    ax.set_xlim(0, 10)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(title, fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.6)
for ax in axes.flat[1:]:
    ax.axvline(S, color="black", lw=1.5, linestyle=":")
axes[1, 0].set_xlabel("$r$"); axes[1, 1].set_xlabel("$r$")

suptitle = fig.suptitle("")


def update_1d(frame):
    s = s_frames[frame]
    lines_1d[0].set_data(x_anim, psi1_phys(s, x_anim))
    lines_1d[1].set_data(r_anim, psi1_comp(s, R_of_r_a))
    lines_1d[2].set_data(r_anim, psi1_hyp(s, k_a))
    lines_1d[3].set_data(r_anim, psi1_hyp(s, k_a))
    suptitle.set_text(f"1D balík:  $T = {s:.1f}$ (nahoře),  $t = {s:.1f}$ (dole)")
    return lines_1d


ani_1d = FuncAnimation(fig, update_1d, frames=len(s_frames), interval=75, blit=False, cache_frame_data=False)
plt.close()
HTML(ani_1d.to_jshtml())

## Animace 3D: mezikroky

Stejná čtveřice pro sférickou vlnu. V prvních třech panelech je vidět $1/R$ úpadek
(a po kompaktifikaci zpomalování a zužování balíku u scri); teprve přeškálování na
$\chi = R\psi$ vrátí signálu na $\mathcal{I}^+$ plnou amplitudu.

In [ ]:
# --- ANIMACE 3D ---
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
titles_3d = [
    "$\\psi(T, R)$ — fyzikální",
    "$\\psi(T, r)$ — jen kompaktifikace",
    "$\\psi(t, r)$ — + hyp. čas",
    "$\\chi(t, r) = R\\psi$ — + přeškálování",
]
ylims_3d = [(-0.05, 0.3), (-0.05, 0.3), (-0.05, 0.3), (-0.05, 1.1)]
lines_3d = []
for ax, title, ylim in zip(axes.flat, titles_3d, ylims_3d):
    (ln,) = ax.plot(x_anim, np.zeros_like(x_anim), lw=2, color="steelblue")
    lines_3d.append(ln)
    ax.set_xlim(0, 10)
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.6)
for ax in axes.flat[1:]:
    ax.axvline(S, color="black", lw=1.5, linestyle=":")
axes[1, 0].set_xlabel("$r$"); axes[1, 1].set_xlabel("$r$")

suptitle = fig.suptitle("")


def update_3d(frame):
    s = s_frames[frame]
    lines_3d[0].set_data(x_anim, psi3_phys(s, x_anim))
    lines_3d[1].set_data(r_anim, psi3_comp(s, R_of_r_a))
    lines_3d[2].set_data(r_anim, psi3_hyp(s, R_of_r_a, k_a, v_geom_a))
    lines_3d[3].set_data(r_anim, chi3_hyp(s, k_a, v_geom_a))
    suptitle.set_text(f"3D balík:  $T = {s:.1f}$ (nahoře),  $t = {s:.1f}$ (dole)")
    return lines_3d


ani_3d = FuncAnimation(fig, update_3d, frames=len(s_frames), interval=75, blit=False, cache_frame_data=False)
plt.close()
HTML(ani_3d.to_jshtml())

## Shrnutí

- **Kompaktifikace sama o sobě nestačí** (1D i 3D): v čase $T$ balík na scri nikdy nedorazí,
  jeho stopa se u $r = S$ nekonečně zužuje a numericky by byla nerozlišitelná.
- **Hyperboloidální čas** to spraví: na řezech $t = \mathrm{konst}$ balík protne $\mathcal{I}^+$
  v konečném čase; signál na scri je $f(t - S)$, tady s maximem v $t = S - R_0 = 6$.
- **Přeškálování** $\chi = R\psi$ je čistě 3D záležitost: kompenzuje geometrický $1/R$ úpadek
  sférické vlny, aby signál na scri nebyl identicky nulový. V 1D žádný úpadek není a $\chi \equiv \psi$.

Numerická evoluce téhož systému (místo exaktní formy) je v `HypSlc_WESystemNumSol.ipynb`.